<a href="https://colab.research.google.com/github/Fespinoza1477/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# TODO
initial_rows = len(df)
df = df.drop_duplicates()
log('TODO 1', 'Dropped duplicate rows', initial_rows - len(df))

[TODO 1] Dropped duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
# TODO
df['price'] = df['price'].astype(str).str.replace('$','', regex=False).astype(float)
log('TODO 2', 'Converted price strings to float', len(df))

[TODO 2] Converted price strings to float (300 row(s))


In [5]:
print((df['qty'] * df['price']).sum())

4594.5


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [6]:
# TODO
initial_rows = len(df)
df = df.dropna(subset=['qty'])
df = df[df['qty'] > 0]
log('TODO 3', 'Dropped missing and negative quantities', initial_rows - len(df))

[TODO 3] Dropped missing and negative quantities (25 row(s))


In [7]:
print((df['qty'] * df['price']).sum())

4740.0


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [8]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger','cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger','foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho', 'rain poncho': 'Rain Poncho'
}
df['item'] = df['item'].map(ITEM_MAP)
log('TODO 4', 'Canonicalized item spelling variants', len(df))

[TODO 4] Canonicalized item spelling variants (275 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [9]:
# TODO
CAT_MAP = {
    'Food': 'Food', 'food':'Food',
    'Merch': 'Merch', 'Apparel': 'Merch',
    'RainGear': 'Rain Gear', 'rain-gear': 'Rain Gear'
}
df['category'] = df['category'].map(CAT_MAP)
log('TODO 5', 'Normalized categories and merged Apparel into Merch', len(df))


[TODO 5] Normalized categories and merged Apparel into Merch (275 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [10]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].isin(['Cheeseburger', 'Foam Finger', 'Rain Poncho']).all()
assert df['category'].isin(['Food', 'Merch', 'Rain Gear']).all()
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [11]:
# TODO
df['revenue'] = df['qty'] * df['price']
category_revenue = df.groupby('category')['revenue'].sum().sort_values(ascending=False)

print("Revenue by Category:")
print(category_revenue.apply(lambda x: f"${x:.2f}"))
print(f"\nOverall Total Revenue : ${df['revenue'].sum():,.2f}")

Revenue by Category:
category
Food         $1656.00
Merch        $1572.00
Rain Gear    $1512.00
Name: revenue, dtype: object

Overall Total Revenue : $4,740.00


**What I would tell the vendor:** _..._

### TODO 8 — read back your log

In [12]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,TODO 1,Dropped duplicate rows,15
1,TODO 2,Converted price strings to float,300
2,TODO 3,Dropped missing and negative quantities,25
3,TODO 4,Canonicalized item spelling variants,275
4,TODO 5,Normalized categories and merged Apparel into ...,275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a). Dropping missing and negative quantities into step 3 changed the revenue the most, taking the calculation from 4594.5 to 4740.0

b). A reasonable person could have chosen to keep 'Apparel' and 'Merch' as separate categories in Step 5 rather than merging them. Had I kept them separate, the reported revenue 'Merch' would have been lower and split across two lines. I chose to merge them because 'Apparel (like foam finger or t-shirts) serves the same operational prupose for the vendor as general mechandise, simplifying the final busines reporting without losing material accuracy.